# A function as a flow rate

Some hazards are easier to write as an ordinary Python function than as a
tree of rate nodes. `defer` is that door: wrap the function, call it with
the time and the parameters it needs, and use the result as a flow rate.

The claim: a recovery rate `0.1 * (1 + amp * sin(t))`, with `amp` a
parameter, empties an infectious compartment along the same curve as the
formula evaluated in NumPy. The two lines should overlie.


## Seasonal recovery

The function below is plain Python. `defer` turns the call into a rate.
Inside the function the arguments are JAX values, so the sine is
`jax.numpy.sin`. The next cell evaluates the rate at each day from the
compiled model and from NumPy. The two lines should overlie.


In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

import jax.numpy as jnp

from summer4 import EntryFlow, FlowModel, Param, Property, PropertyData, PropertyMap, Time, defer


def recovery(t, amp):
    """People per day, for one infectious person, with a seasonal bump."""
    return 0.1 * (1.0 + amp * jnp.sin(2.0 * jnp.pi * t / 365.0))


state = Property("state", ("I",))
pmap = PropertyMap.from_property(state)
model = FlowModel(pmap)
model.add_flow(EntryFlow("in", state["I"], defer(recovery)(Time(), Param("amp"))))
compiled = model.compile()
y0 = PropertyData.wrap(pmap, np.array([0.0]))
days = np.linspace(0.0, 365.0, 366)
amp = 0.4


def model_rate(t: float) -> float:
    return float(np.asarray(compiled.vector_field(t, y0, {"amp": amp}).data)[0])


modelled = np.array([model_rate(float(t)) for t in days])
formula = 0.1 * (1.0 + amp * np.sin(2.0 * np.pi * days / 365.0))
np.testing.assert_allclose(modelled, formula, rtol=1e-5, atol=1e-5)

pd.DataFrame({"model": modelled, "numpy formula": formula}, index=days).plot(
    title="Seasonal recovery rate from defer(recovery) matches NumPy",
    labels={"index": "day", "value": "per person per day"},
)
